#Домашнее задание 2

In [9]:
import pandas as pd
import re

df = pd.read_csv("train.csv")

print("Информация о датасете:")
df.info()
print()

print("Пропуски:")
print(df.isna().sum(), "\n")

print("Описательная статистика:")
print(df.describe(include="all"), "\n")


survival_by_class = df.groupby("Pclass")["Survived"].mean() * 100
print("Процент выживаемости по классам:")
print(survival_by_class, "\n")


def extract_first_name(full_name):
    parts = full_name.split(", ", 1)
    rest = parts[1] if len(parts) > 1 else full_name

    rest = re.sub(r"^[A-Za-z]+\. ", "", rest)

    bracket = re.search(r"\((.*?)\)", rest)
    if bracket:
        return bracket.group(1).split()[0]

    rest = re.sub(r"\(.*?\)", "", rest).strip()
    return rest.split()[0] if rest else None


df["FirstName"] = df["Name"].apply(extract_first_name)

male_name = df[df["Sex"] == "male"]["FirstName"].mode().iloc[0]
female_name = df[df["Sex"] == "female"]["FirstName"].mode().iloc[0]

print("Самое популярное мужское имя:", male_name)
print("Самое популярное женское имя:", female_name, "\n")


print("Самые популярные имена по классам:")
for pclass in sorted(df["Pclass"].unique()):
    male_mode = df[(df["Pclass"] == pclass) & (df["Sex"] == "male")]["FirstName"].mode().iloc[0]
    female_mode = df[(df["Pclass"] == pclass) & (df["Sex"] == "female")]["FirstName"].mode().iloc[0]
    print(f"Pclass = {pclass}: мужское = {male_mode}, женское = {female_mode}")
print()


print("Пассажиры старше 44 лет:")
print(df[df["Age"] > 44], "\n")


print("Пассажиры младше 44 лет мужского пола:")
print(df[(df["Age"] < 44) & (df["Sex"] == "male")], "\n")


cab = df[["PassengerId", "Cabin"]].dropna().copy()
cab["Cabin"] = cab["Cabin"].str.split()
exploded = cab.explode("Cabin")

cabin_occupancy = exploded.groupby("Cabin")["PassengerId"].nunique()

n_seat_cabins = cabin_occupancy.value_counts().sort_index()

print("Количество n-местных кабин:")
print(n_seat_cabins, "\n")


no_relatives = df[(df["SibSp"] == 0) & (df["Parch"] == 0)].shape[0]
print("Количество пассажиров без родственников на борту:", no_relatives)

Информация о датасете:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB

Пропуски:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
E